# Smart Hangman Solver (uses `words.csv`)

A frequency-based solver: at each step it keeps only the dictionary words that fit the revealed pattern and picks the letter appearing in the most of them.

We split the words 80/20 so we can measure how well it does on words it has **not** seen.

## 1. Load data & train/test split

In [ ]:
import random
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt

# Colab: upload words.csv first ->  from google.colab import files; files.upload()
df = pd.read_csv("words.csv", keep_default_na=False)  # keeps words like "null" and "none" as text
words = df[df["hangman_ok"] == "yes"]["word"].tolist()
print(len(words), "usable words")

# Train/test split: the solver only "learns" from the train words
random.seed(42)
random.shuffle(words)
split = int(0.8 * len(words))
train_words, test_words = words[:split], words[split:]
print(len(train_words), "train /", len(test_words), "test")

## 2. The solver

In [ ]:
# Group the training dictionary by word length for fast lookup
by_length = defaultdict(list)
for w in train_words:
    by_length[len(w)].append(w)

overall_freq = Counter("".join(train_words))

def matches(word, pattern, wrong):
    """True if `word` fits the revealed pattern and contains no wrong letter."""
    if len(word) != len(pattern):
        return False
    for w_ch, p_ch in zip(word, pattern):
        if p_ch != "_" and w_ch != p_ch:
            return False
        if p_ch == "_" and w_ch in pattern:   # a revealed letter can't hide elsewhere
            return False
    return not any(ch in word for ch in wrong)

def next_guess(pattern, guessed, wrong):
    """Pick the unguessed letter that appears in the most candidate words."""
    candidates = [w for w in by_length.get(len(pattern), []) if matches(w, pattern, wrong)]
    counts = Counter()
    for w in candidates:
        for ch in set(w) - guessed:
            counts[ch] += 1
    if counts:
        return counts.most_common(1)[0][0]
    # No candidate left (unseen word): fall back to overall letter frequency
    for ch, _ in overall_freq.most_common():
        if ch not in guessed:
            return ch

def play(word, max_wrong=6, verbose=False):
    guessed, wrong = set(), set()
    pattern = "_" * len(word)
    while len(wrong) < max_wrong and "_" in pattern:
        g = next_guess(pattern, guessed, wrong)
        guessed.add(g)
        if g in word:
            pattern = "".join(c if c in guessed else "_" for c in word)
        else:
            wrong.add(g)
        if verbose:
            print(f"guess {g.upper()}  ->  {' '.join(pattern.upper())}   wrong={len(wrong)}")
    return "_" not in pattern, len(wrong)

## 3. Demo game

In [ ]:
# Watch the solver play one unseen word
play(random.choice(test_words), verbose=True)

## 4. Evaluate on unseen words

In [ ]:
sample = random.sample(test_words, 500)
results = [(w, *play(w)) for w in sample]

wins = sum(r[1] for r in results)
print(f"Win rate on unseen words: {wins/len(results):.1%}")
print(f"Avg wrong guesses:        {sum(r[2] for r in results)/len(results):.2f} / 6")

## 5. Win rate by word length

In [ ]:
by_len = defaultdict(list)
for w, won, _ in results:
    by_len[len(w)].append(won)

lengths = sorted(l for l in by_len if len(by_len[l]) >= 5)
rates = [sum(by_len[l]) / len(by_len[l]) for l in lengths]

plt.bar(lengths, rates)
plt.xlabel("Word length"); plt.ylabel("Win rate"); plt.ylim(0, 1)
plt.title("Solver win rate by word length (unseen words)")
plt.show()

## Ideas to extend
- Add letter-pair (bigram) statistics for words not in the dictionary
- Weight words by their rank in the frequency list
- Compare with a baseline that guesses letters in fixed `etaoin` order